# Feature Selection
#### 최적 모델로 선택된 CatBoost 기준 feature selection 진행

In [16]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score
)

from catboost import CatBoostClassifier

In [2]:
# =========================
# 데이터 지정 (Cleaning)
# =========================
DATA_DIR = Path('../Data')
train_df = pd.read_csv(DATA_DIR / "train_cleaning.csv")
test_df = pd.read_csv(DATA_DIR / "test_cleaning.csv")

target_col = "fraud"   # 필요하면 "TARGET" 등으로 수정

# =========================
# Train / Test 분리
# =========================

X = train_df.drop(columns=[target_col]).copy()
y = train_df[target_col].copy()

X_test = test_df.copy()   # test에는 target 없음

print("Train X shape:", X.shape)
print("Train y shape:", y.shape)
print("Test X shape :", X_test.shape)
print("Positive ratio:", y.mean().round(4))

Train X shape: (18000, 125)
Train y shape: (18000,)
Test X shape : (12000, 125)
Positive ratio: 0.1582


In [3]:
def recall_at_k(y_true, y_prob, top_ratio=0.10):
    y_true = np.array(y_true)
    y_prob = np.array(y_prob)

    n_top = max(1, int(len(y_prob) * top_ratio))
    top_idx = np.argsort(y_prob)[::-1][:n_top]

    total_positive = y_true.sum()
    if total_positive == 0:
        return 0.0

    return y_true[top_idx].sum() / total_positive

In [4]:
def get_catboost_oof_and_importance(
    X, y,
    n_splits=5,
    random_state=42,
    cat_params=None
):
    if cat_params is None:
        cat_params = {
            "loss_function": "Logloss",
            "eval_metric": "AUC",
            "iterations": 10000,
            "learning_rate": 0.03,
            "depth": 6,
            "l2_leaf_reg": 0.5,
            "bootstrap_type": "Bernoulli",
            "subsample": 0.8,
            "random_seed": random_state,
            "verbose": 0
        }

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    oof_pred = np.zeros(len(X))
    importance_list = []
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
        print(f"\n{'='*20} Fold {fold} {'='*20}")

        X_train = X.iloc[train_idx].copy()
        X_valid = X.iloc[valid_idx].copy()
        y_train = y.iloc[train_idx].copy()
        y_valid = y.iloc[valid_idx].copy()

        neg = (y_train == 0).sum()
        pos = (y_train == 1).sum()
        scale_pos_weight = (neg / pos) * 1.1 if pos > 0 else 1.0

        # fold별 파라미터 복사 후 scale_pos_weight 주입
        fold_cat_params = cat_params.copy()
        fold_cat_params["scale_pos_weight"] = scale_pos_weight

        model = CatBoostClassifier(**fold_cat_params)

        model.fit(
            X_train, y_train,
            eval_set=(X_valid, y_valid),
            verbose=False
        )

        valid_prob = model.predict_proba(X_valid)[:, 1]
        oof_pred[valid_idx] = valid_prob

        fold_auc = roc_auc_score(y_valid, valid_prob)
        fold_pr = average_precision_score(y_valid, valid_prob)

        fold_scores.append({
            "fold": fold,
            "roc_auc": fold_auc,
            "pr_auc": fold_pr,
            "scale_pos_weight": scale_pos_weight
        })

        fold_importance = pd.DataFrame({
            "feature": X.columns,
            "importance": model.get_feature_importance()
        })
        fold_importance["fold"] = fold
        importance_list.append(fold_importance)

        print(
            f"ROC-AUC: {fold_auc:.4f} | "
            f"PR-AUC: {fold_pr:.4f} | "
            f"scale_pos_weight: {scale_pos_weight:.4f}"
        )

    importance_df = pd.concat(importance_list, axis=0, ignore_index=True)
    fold_score_df = pd.DataFrame(fold_scores)

    return oof_pred, importance_df, fold_score_df

In [5]:
cat_oof_full, importance_raw_df, fold_score_full_df = get_catboost_oof_and_importance(
    X=X,
    y=y,
    n_splits=5,
    random_state=2542
)


==================== Fold 1 ====================
ROC-AUC: 0.6826 | PR-AUC: 0.2832 | scale_pos_weight: 5.8504

==================== Fold 2 ====================
ROC-AUC: 0.7275 | PR-AUC: 0.3358 | scale_pos_weight: 5.8504

==================== Fold 3 ====================
ROC-AUC: 0.6965 | PR-AUC: 0.2820 | scale_pos_weight: 5.8535

==================== Fold 4 ====================
ROC-AUC: 0.7053 | PR-AUC: 0.2972 | scale_pos_weight: 5.8535

==================== Fold 5 ====================
ROC-AUC: 0.7004 | PR-AUC: 0.2826 | scale_pos_weight: 5.8535


In [6]:
baseline_result = {
    "feature_set": "all_features",
    "n_features": X.shape[1],
    "roc_auc": roc_auc_score(y, cat_oof_full),
    "pr_auc": average_precision_score(y, cat_oof_full),
    "recall_top5": recall_at_k(y, cat_oof_full, 0.05),
    "recall_top10": recall_at_k(y, cat_oof_full, 0.10),
    "recall_top20": recall_at_k(y, cat_oof_full, 0.20),
}

baseline_result

{'feature_set': 'all_features',
 'n_features': 125,
 'roc_auc': 0.7017906283740493,
 'pr_auc': 0.29276555168245966,
 'recall_top5': np.float64(0.12956460674157302),
 'recall_top10': np.float64(0.22612359550561797),
 'recall_top20': np.float64(0.3886938202247191)}

In [7]:
importance_summary = (
    importance_raw_df
    .groupby("feature", as_index=False)["importance"]
    .mean()
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

importance_summary["rank"] = np.arange(1, len(importance_summary) + 1)

print(importance_summary.head(20))

                         feature  importance  rank
0               accident_parking   12.462185     1
1                  age_of_driver    6.287579     2
2             high_education_ind    5.611052     3
3               age_of_driver_sq    5.417321     4
4                 marital_status    2.999554     5
5             address_change_ind    2.995242     6
6                 witness_absent    2.838648     7
7                no_verification    2.580654     8
8                     claim_year    1.962004     9
9    zip_rate_x_accident_highway    1.830797    10
10  age_of_driver_x_safty_rating    1.705563    11
11       female_x_witness_absent    1.481751    12
12  vehicle_price_per_driver_age    1.329596    13
13                  safty_rating    1.319814    14
14      safty_rating_x_liab_prct    1.221548    15
15               age_over_safety    1.157159    16
16     age_of_driver_x_liab_prct    1.124942    17
17             safety_risk_score    0.977114    18
18               day_of_week_si

In [8]:
importance_summary.head(30)

,feature,importance,rank
0,accident_parking,12.462185,1
1,age_of_driver,6.287579,2
2,high_education_ind,5.611052,3
3,age_of_driver_sq,5.417321,4
4,marital_status,2.999554,5
5,address_change_ind,2.995242,6
6,witness_absent,2.838648,7
7,no_verification,2.580654,8
8,claim_year,1.962004,9
9,zip_rate_x_accident_highway,1.830797,10


In [9]:
def evaluate_feature_subsets(
    X, y,
    importance_summary,
    feature_counts=[20, 30, 40, 50, 60, 70, 80, 100],
    n_splits=5,
    random_state=42,
    cat_params=None
):
    results = []

    # 전체 피처도 포함
    all_features = list(X.columns)
    feature_sets = [("all_features", all_features)]

    for n in feature_counts:
        selected = importance_summary["feature"].head(n).tolist()
        feature_sets.append((f"top_{n}", selected))

    for name, selected_features in feature_sets:
        print(f"\n{'#'*60}")
        print(f"Feature Set: {name} | n_features = {len(selected_features)}")
        print(f"{'#'*60}")

        X_sub = X[selected_features].copy()

        oof_pred, _, _ = get_catboost_oof_and_importance(
            X=X_sub,
            y=y,
            n_splits=n_splits,
            random_state=random_state,
            cat_params=cat_params
        )

        row = {
            "feature_set": name,
            "n_features": len(selected_features),
            "roc_auc": roc_auc_score(y, oof_pred),
            "pr_auc": average_precision_score(y, oof_pred),
            "recall_top5": recall_at_k(y, oof_pred, 0.05),
            "recall_top10": recall_at_k(y, oof_pred, 0.10),
            "recall_top20": recall_at_k(y, oof_pred, 0.20),
        }
        results.append(row)

    results_df = pd.DataFrame(results).sort_values(
        by=["pr_auc", "roc_auc"], ascending=False
    ).reset_index(drop=True)

    return results_df

In [10]:
subset_results_df = evaluate_feature_subsets(
    X=X,
    y=y,
    importance_summary=importance_summary,
    feature_counts=[20, 30, 40, 50, 60, 70, 80, 100],
    n_splits=5,
    random_state=42
)

subset_results_df.round(4)


############################################################
Feature Set: all_features | n_features = 125
############################################################

==================== Fold 1 ====================
ROC-AUC: 0.6769 | PR-AUC: 0.2641 | scale_pos_weight: 5.8504

==================== Fold 2 ====================
ROC-AUC: 0.7073 | PR-AUC: 0.3165 | scale_pos_weight: 5.8504

==================== Fold 3 ====================
ROC-AUC: 0.6977 | PR-AUC: 0.2817 | scale_pos_weight: 5.8535

==================== Fold 4 ====================
ROC-AUC: 0.7195 | PR-AUC: 0.3112 | scale_pos_weight: 5.8535

==================== Fold 5 ====================
ROC-AUC: 0.7085 | PR-AUC: 0.2967 | scale_pos_weight: 5.8535

############################################################
Feature Set: top_20 | n_features = 20
############################################################

==================== Fold 1 ====================
ROC-AUC: 0.6779 | PR-AUC: 0.2667 | scale_pos_weight: 5.8504

==========

,feature_set,n_features,roc_auc,pr_auc,recall_top5,recall_top10,recall_top20
0,top_20,20,0.7038,0.2974,0.1289,0.2317,0.3904
1,top_40,40,0.7036,0.2954,0.1306,0.2279,0.3940
2,top_30,30,0.7037,0.2952,0.1334,0.2296,0.3908
3,all_features,125,0.7013,0.2894,0.1254,0.2289,0.3855
4,top_80,80,0.7012,0.2892,0.1236,0.2268,0.3880
5,top_100,100,0.7010,0.2890,0.1246,0.2251,0.3887
6,top_70,70,0.7008,0.2889,0.1215,0.2216,0.3848
7,top_50,50,0.7008,0.2885,0.1268,0.2219,0.3915
8,top_60,60,0.7011,0.2861,0.1229,0.2237,0.3940


In [11]:
best_row = subset_results_df.iloc[0]
best_feature_set_name = best_row["feature_set"]

print("Best Feature Set:", best_feature_set_name)
print(best_row)

Best Feature Set: top_20
feature_set       top_20
n_features            20
roc_auc         0.703763
pr_auc          0.297356
recall_top5     0.128862
recall_top10    0.231742
recall_top20    0.390449
Name: 0, dtype: object


In [12]:
if best_feature_set_name == "all_features":
    best_features = list(X.columns)
else:
    best_n = int(best_feature_set_name.split("_")[1])
    best_features = importance_summary["feature"].head(best_n).tolist()

print("Number of selected features:", len(best_features))
print(best_features)

Number of selected features: 20
['accident_parking', 'age_of_driver', 'high_education_ind', 'age_of_driver_sq', 'marital_status', 'address_change_ind', 'witness_absent', 'no_verification', 'claim_year', 'zip_rate_x_accident_highway', 'age_of_driver_x_safty_rating', 'female_x_witness_absent', 'vehicle_price_per_driver_age', 'safty_rating', 'safty_rating_x_liab_prct', 'age_over_safety', 'age_of_driver_x_liab_prct', 'safety_risk_score', 'day_of_week_sin', 'female_x_liab_prct']


In [13]:
# -------------------------
# Top20
# -------------------------
top20_features = importance_summary["feature"].head(20).tolist()

train_selection_top20 = train_df[top20_features + [target_col]].copy()
test_selection_top20 = test_df[top20_features].copy()

# -------------------------
# Top30
# -------------------------
top30_features = importance_summary["feature"].head(30).tolist()

train_selection_top30 = train_df[top30_features + [target_col]].copy()
test_selection_top30 = test_df[top30_features].copy()

# -------------------------
# Top40
# -------------------------
top40_features = importance_summary["feature"].head(40).tolist()

train_selection_top40 = train_df[top40_features + [target_col]].copy()
test_selection_top40 = test_df[top40_features].copy()

In [14]:
print("Top20 Train:", train_selection_top20.shape)
print("Top20 Test :", test_selection_top20.shape)

print("Top30 Train:", train_selection_top30.shape)
print("Top30 Test :", test_selection_top30.shape)

print("Top40 Train:", train_selection_top40.shape)
print("Top40 Test :", test_selection_top40.shape)

Top20 Train: (18000, 21)
Top20 Test : (12000, 20)
Top30 Train: (18000, 31)
Top30 Test : (12000, 30)
Top40 Train: (18000, 41)
Top40 Test : (12000, 40)


In [15]:
train_selection_top20.to_csv(DATA_DIR / "train_selection_top20.csv", index=False)
test_selection_top20.to_csv(DATA_DIR / "test_selection_top20.csv", index=False)

train_selection_top30.to_csv(DATA_DIR / "train_selection_top30.csv", index=False)
test_selection_top30.to_csv(DATA_DIR / "test_selection_top30.csv", index=False)

train_selection_top40.to_csv(DATA_DIR / "train_selection_top40.csv", index=False)
test_selection_top40.to_csv(DATA_DIR / "test_selection_top40.csv", index=False)

### 최종 데이터셋 선택

In [18]:
target_col = "fraud"

datasets = {
    "top20": pd.read_csv(DATA_DIR / "train_selection_top20.csv"),
    "top30": pd.read_csv(DATA_DIR / "train_selection_top30.csv"),
    "top40": pd.read_csv(DATA_DIR / "train_selection_top40.csv"),
}

In [19]:
cat_params = {
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "iterations": 300,
    "learning_rate": 0.03,
    "depth": 4,
    "l2_leaf_reg": 5,
    "subsample": 1.0,
    "bootstrap_type": "Bernoulli",
    "random_seed": 42,
    "auto_class_weights": "Balanced",
    "verbose": -1
}

In [21]:
def recall_at_k(y_true, y_prob, ratio):
    n = int(len(y_prob) * ratio)
    idx = np.argsort(y_prob)[::-1][:n]
    return y_true.iloc[idx].sum() / y_true.sum()

def precision_at_k(y_true, y_prob, ratio):
    n = int(len(y_prob) * ratio)
    idx = np.argsort(y_prob)[::-1][:n]
    return y_true.iloc[idx].mean()

def run_oof(X, y):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof = np.zeros(len(X))

    for train_idx, valid_idx in skf.split(X, y):
        model = CatBoostClassifier(**cat_params)
        model.fit(X.iloc[train_idx], y.iloc[train_idx], verbose=False)

        prob = model.predict_proba(X.iloc[valid_idx])[:, 1]
        oof[valid_idx] = prob

    return oof

In [22]:
def find_best_threshold(y, oof):
    thresholds = np.arange(0.1, 0.9, 0.01)
    best_f1 = -1
    best_thr = 0.5

    for t in thresholds:
        pred = (oof >= t).astype(int)
        f1 = f1_score(y, pred)

        if f1 > best_f1:
            best_f1 = f1
            best_thr = t

    return best_thr, best_f1

In [23]:
results = []

for name, df in datasets.items():
    print(f"\n===== {name} =====")

    X = df.drop(columns=[target_col])
    y = df[target_col]

    oof = run_oof(X, y)

    roc = roc_auc_score(y, oof)
    pr = average_precision_score(y, oof)

    # threshold 최적화
    best_thr, best_f1 = find_best_threshold(y, oof)

    pred = (oof >= best_thr).astype(int)

    precision = precision_score(y, pred)
    recall = recall_score(y, pred)

    # Top-K
    r5 = recall_at_k(y, oof, 0.05)
    r10 = recall_at_k(y, oof, 0.10)
    r20 = recall_at_k(y, oof, 0.20)

    p10 = precision_at_k(y, oof, 0.10)

    results.append({
        "feature_set": name,
        "n_features": X.shape[1],
        "roc_auc": roc,
        "pr_auc": pr,
        "best_threshold": best_thr,
        "f1": best_f1,
        "precision": precision,
        "recall": recall,
        "recall_top5": r5,
        "recall_top10": r10,
        "recall_top20": r20,
        "precision_top10": p10
    })


===== top20 =====

===== top30 =====

===== top40 =====


In [24]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by=["pr_auc", "recall_top10"],
    ascending=False
).reset_index(drop=True)

results_df.round(4)

,feature_set,n_features,roc_auc,pr_auc,best_threshold,f1,precision,recall,recall_top5,recall_top10,recall_top20,precision_top10
0,top30,30,0.7075,0.2991,0.54,0.3669,0.2734,0.5576,0.1303,0.2310,0.3904,0.3656
1,top20,20,0.7065,0.2983,0.54,0.3684,0.2732,0.5657,0.1348,0.2293,0.3919,0.3628
2,top40,40,0.7066,0.2975,0.54,0.3664,0.2744,0.5513,0.1296,0.2275,0.3940,0.3600


- feature_selection_top30으로 최종 데이터셋 선정